# 04. 공통 평가 질문 실행

OpenAI와 Local Retriever 완성 후 같은 질문 세트를 실행해 비교합니다. 이 노트북은 실행 결과를 포함하지 않습니다.


In [9]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

In [14]:
import json
import sys
import time
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "api_main.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
QUESTION_PATH = PROJECT_ROOT / "data" / "evaluation" / "questions.jsonl"

with QUESTION_PATH.open(encoding="utf-8") as file:
    questions = [json.loads(line) for line in file]

display(pd.DataFrame(questions)[["question_id", "type", "question", "expected_doc_ids"]])


,question_id,type,question,expected_doc_ids
0,Q01,single,한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화 사업의 예...,[doc_001]
1,Q02,numeric,"한국연구재단 UICC 기능개선 사업의 전체 사업비와 SW 개발비를 알려주고, SW ...",[doc_002]
2,Q03,single,한국연구재단 UICC 기능개선 사업에서 데이터 입력 기능은 무엇을 개선해야 하나?,[doc_002]
3,Q04,single,스포츠윤리센터 LMS 기능개선 사업의 교육 미디어 플레이어 요구사항을 정리해줘.,[doc_009]
4,Q05,table,"서울시립대학교 사업에서 대입전형 자료, 학적 정보, 학생 활동 정보는 각각 어느 부...",[doc_013]
5,Q06,table,서울시립대학교의 대학생활만족도 정보는 어느 부서가 담당하고 어떤 정보인가?,[doc_013]
6,Q07,multi_document,한영대학교 트랙운영 사업과 한국연구재단 UICC 사업의 예산과 수행 기간을 비교하고...,"[doc_001, doc_002]"
7,Q08,unsupported,한국연구재단 UICC 기능개선 사업에서 블록체인 도입을 필수 요구사항으로 명시했나?,[]
8,Q09,follow_up,스포츠윤리센터 LMS 기능개선 사업의 미디어 플레이어 요구사항을 알려줘.,[doc_009]
9,Q10,follow_up,그 요구사항 중 자막 기능도 포함되어 있나?,[doc_009]


In [11]:
# OpenAI와 Local 구현 완료 후 아래 셀을 실행합니다.
sys.path.insert(0, str(PROJECT_ROOT))
from api_main import run

CONFIG_PATH = PROJECT_ROOT / "config" / "default.yaml"

PROFILE = "openai"  # 실행 시 openai 또는 local 중 하나를 선택

records = []
history_by_conversation = {}
for item in questions:
    conversation_id = item.get("conversation_id")
    history = history_by_conversation.get(conversation_id)
    started = time.perf_counter()
    response = run(
        item["question"],
        config_path=str(CONFIG_PATH),
        profile=PROFILE,
        filters=item.get("filters"),
        history=history,
    )
    elapsed = time.perf_counter() - started
    if conversation_id:
        history_by_conversation.setdefault(conversation_id, []).append(
            {"question": item["question"], "answer": response["answer"]}
        )
    retrieved = [source.get("doc_id") for source in response["sources"]]
    answer = response["answer"]
    expected = item.get("expected_doc_ids", [])
    keywords = item.get("keywords", [])
    records.append({
        "question_id": item["question_id"],
        "type": item["type"],
        "profile": PROFILE,
        "question": item["question"],
        "retrieved_doc_ids": retrieved,
        # 정답 문서를 검색했는지 (expected_doc_ids가 없는 unsupported 유형은 None)
        "doc_hit": bool(set(expected) & set(retrieved)) if expected else None,
        # 기준 답변의 키워드가 최종 답변에 몇 개 포함됐는지
        "keyword_hit": sum(1 for k in keywords if k in answer),
        "keyword_total": len(keywords),
        # 답변이 비어 있는지 (reasoning 토큰 소진 등으로 생성 실패)
        "is_empty": not answer.strip(),
        "final_answer": answer,
        "response_seconds": round(elapsed, 3),
    })

results = pd.DataFrame(records)
display(results.drop(columns=["question", "final_answer"]))


,question_id,type,profile,retrieved_doc_ids,doc_hit,keyword_hit,keyword_total,is_empty,response_seconds
0,Q01,single,openai,"[doc_001, doc_043, doc_008, doc_008, doc_001]",True,3,3,False,8.476
1,Q02,numeric,openai,"[doc_017, doc_002, doc_040, doc_061, doc_075]",True,1,2,False,12.436
2,Q03,single,openai,"[doc_002, doc_002, doc_011, doc_022, doc_041]",True,0,3,False,14.150
3,Q04,single,openai,"[doc_009, doc_009, doc_009, doc_009, doc_055]",True,3,3,False,14.446
4,Q05,table,openai,"[doc_013, doc_013, doc_013, doc_013, doc_013]",True,3,6,False,16.400
5,Q06,table,openai,"[doc_013, doc_013, doc_013, doc_013, doc_013]",True,3,3,False,12.838
6,Q07,multi_document,openai,"[doc_002, doc_045, doc_042, doc_001, doc_073]",True,2,4,False,14.898
7,Q08,unsupported,openai,"[doc_002, doc_041, doc_027, doc_075, doc_038]",None,2,2,False,10.465
8,Q09,follow_up,openai,"[doc_009, doc_009, doc_009, doc_011, doc_027]",True,2,2,False,11.392
9,Q10,follow_up,openai,"[doc_009, doc_009, doc_055, doc_027, doc_005]",True,2,2,False,10.634


In [13]:
# 결과 저장과 요약. 전체 답변이 잘리지 않도록 CSV로 남긴다.
RESULT_PATH = QUESTION_PATH.parent / f"results_{PROFILE}.csv"
results.to_csv(RESULT_PATH, index=False, encoding="utf-8-sig")
print(f"결과 저장: {RESULT_PATH}")

print(f"\n전체 {len(results)}문항")
print(f"  정답 문서 적중 : {results['doc_hit'].sum():.0f} / {results['doc_hit'].notna().sum()}")
print(f"  키워드 포함    : {results['keyword_hit'].sum()} / {results['keyword_total'].sum()}")
print(f"  빈 답변        : {results['is_empty'].sum()}건 {list(results.loc[results['is_empty'], 'question_id'])}")
print(f"  평균 응답 시간  : {results['response_seconds'].mean():.1f}초")

print("\n[유형별]")
summary = results.groupby("type").agg(
    문항수=("question_id", "count"),
    문서적중=("doc_hit", "sum"),
    키워드적중=("keyword_hit", "sum"),
    키워드전체=("keyword_total", "sum"),
    빈답변=("is_empty", "sum"),
    평균초=("response_seconds", "mean"),
).round(1)
display(summary)

결과 저장: /home/seoho/sprint-ai-mid-project_team3/data/evaluation/results_openai.csv

전체 13문항
  정답 문서 적중 : 12 / 12
  키워드 포함    : 22 / 37
  빈 답변        : 1건 ['Q11']
  평균 응답 시간  : 12.6초

[유형별]


,문항수,문서적중,키워드적중,키워드전체,빈답변,평균초
type,,,,,,
follow_up,2,2,4,4,0,11.0
image,3,3,1,7,1,12.8
multi_document,1,True,2,4,0,14.9
numeric,1,True,1,2,0,12.4
single,3,3,6,9,0,12.4
table,2,2,6,9,0,14.6
unsupported,1,0,2,2,0,10.5


실행 결과는 Google Sheets 개발 기록에 질문 ID, 설정값, 검색 문서, 최종 답변, 응답 시간을 기록합니다. 숫자 계산과 표 기반 질문은 reference_answer와 직접 비교해 사람이 최종 판정합니다.
